<a href="https://colab.research.google.com/github/Ragul2526/stable-diffusion-anime-finetune-lora/blob/main/anime_lora_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#One thing to remember if you are saving the file in github , always go to Edit->clear all output.
##Since i have used tqdm you will get invalid notebook error in github. Save after clearing the outputs.

In [ ]:
!pip install diffusers transformers accelerate datasets peft -q

In [ ]:
#These are the libraries needed for the fine tuning of the anime dataset
import torch
import numpy as np
from datasets import load_dataset
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
#Check if gpu available or not go to view resources and change to T4 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
#It will take about 15 to 30 minutes to download based on the internet speed.
dataset = load_dataset("huggan/anime-faces", split="train", streaming=True)
dataset = dataset.take(5000)
print("Dataset ready!")

In [ ]:
samples = []
for item in dataset.take(10):
    samples.append(item["image"])
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Sample Anime Face Images", fontsize=14)

for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples[i])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import Dataset
from tqdm import tqdm
transform = transforms.Compose([
    transforms.Resize((512,512)),
    transforms.ToTensor(), #to convert from [0,255] -> [0,1]
    transforms.Normalize([0.5,0.5,0.5],
                         [0.5,0.5,0.5]) #normalizing the values from [0,1] -> [-1,1]

])
class AnimeDataset(Dataset):
  def __init__(self, hf_dataset, tokenizer, caption = "anime face, high quality, detailed") :

    self.items = []
    self.tokenizer = tokenizer
    self.caption =  caption
    for item in tqdm(hf_dataset, desc="Loading images", unit="img"):
            self.items.append(item["image"])
  def __len__(self):
    return len(self.items)
  def __getitem__(self, idx):
     image = self.items[idx].convert("RGB")
     image = transform(image)

     token = self.tokenizer(
         self.caption,
         padding = "max_length",
         max_length = self.tokenizer.model_max_length,
         truncation = True,
         return_tensors = "pt"
     )
     return {
         "pixel_values": image,
          "input_ids": token.input_ids.squeeze()
     }



#This will take above 10 minutes so just run and chill for a while 😭

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
anime_dataset = AnimeDataset(dataset,tokenizer)
dataloader = DataLoader(anime_dataset, batch_size = 2, shuffle = True) #you can change the batch size to 8 if you have more vram , I am limited to 15gb from colab.
print(f"Total images : {len(anime_dataset)}")
print(f"Total batches: {len(dataloader)}")

batch =  next(iter(dataloader))
print(f"Image batch shape: {batch["pixel_values"].shape}")
print(f"Token batch shape : {batch["input_ids"].shape}")

In [ ]:
from diffusers import AutoencoderKL
from transformers import CLIPTextModel

print("Loading models...")

vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder")
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

vae = vae.to(device)
text_encoder = text_encoder.to(device)
unet = unet.to(device)

print("Models loaded!")

In [ ]:
#!pip install torchao --upgrade -q #if you got "Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported" while running, uncomment these two lines and run them
#!pip install peft --upgrade -q



lora_config = LoraConfig(
    r=4,                          # rank -> higher = more expressive but slower
    lora_alpha=32,                # scaling factor
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],  # attention layers
    lora_dropout=0.1,
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

In [ ]:
#uncomment the below code if you have already trained the model and saved to your gdrive and want to finetune
'''from peft import PeftModel

load_path = "/content/drive/MyDrive/lora-anime-output"

# Load base model
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet")
unet = PeftModel.from_pretrained(unet, load_path)
unet = unet.to(device)

print("LoRA weights loaded from Drive!")
unet.print_trainable_parameters()'''

In [ ]:
from diffusers import DDPMScheduler
import torch.nn.functional as F

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

NUM_EPOCHS = 1 #why just 1? its enough to see anime style  and it takes too long in colab, one epoch seems to take 1.5 hours so its better to have one epoch. if you are running it in your local system you can increase epoch ->3 or 5 or 10

In [ ]:
from tqdm import tqdm

unet.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0
    progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", unit="batch")

    for batch in progress:
        pixel_values = batch["pixel_values"].to(device)
        input_ids    = batch["input_ids"].to(device)


        with torch.no_grad():
            latents      = vae.encode(pixel_values).latent_dist.sample()
            latents      = latents * vae.config.scaling_factor
            text_embeds  = text_encoder(input_ids)[0]


        noise     = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)


        noise_pred = unet(noisy_latents, timesteps, text_embeds).sample


        loss = F.mse_loss(noise_pred, noise)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/lora-anime-output"
os.makedirs(save_path, exist_ok=True)
unet.save_pretrained(save_path)
print(f"LoRA weights saved to Google Drive: {save_path}")

In [ ]:
from diffusers import StableDiffusionPipeline
base_pipe =StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device)

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device)

# Load LoRA weights into pipeline
unet = unet.to(torch.float16)
pipe.unet = unet
pipe.unet.eval()

print("Pipelines are ready!")

In [ ]:
prompts = [
    "anime face, beautiful girl, blue hair, high quality",
    "anime face, boy with silver hair, detailed eyes",
    "anime face, girl with red eyes, fantasy style",
    "anime face, warrior with dark hair, epic lighting"
]
fig, axes = plt.subplots(len(prompts), 2, figsize=(10, 5 * len(prompts)))
fig.suptitle("Base SD vs LoRA Fine-tuned", fontsize=14)

for i, prompt in enumerate(prompts):
    print(f"Generating prompt {i+1}...")

    with torch.no_grad():
        base_image = base_pipe(
            prompt,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]

        lora_image = pipe(
            prompt,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]

    axes[i][0].imshow(base_image)
    axes[i][0].set_title("Base Model", fontsize=10)
    axes[i][0].axis("off")

    axes[i][1].imshow(lora_image)
    axes[i][1].set_title("LoRA Fine-tuned", fontsize=10)
    axes[i][1].axis("off")

plt.tight_layout()
plt.show()

## Results

### Base SD vs LoRA Fine-tuned Comparison

The LoRA fine-tuned model shows clear style transfer towards anime aesthetics:
- Larger, more expressive eyes
- Softer facial features
- Characteristic anime color palette

### Observations
- Trained for 1 epoch on 5000 images from `huggan/anime-faces`
- Base model generates realistic anime art style
- LoRA model shifts towards chibi/moe anime style from the dataset

### Limitations
- Slight blurriness due to low resolution (64×64) source images
- More epochs (3-5) and higher resolution dataset would improve sharpness

### Future improvements
- Train on higher resolution dataset
- Increase epochs to 3-5
- Experiment with higher LoRA rank (r=8 or r=16)